# Arsenal Analytics Dashboard

Data source: StatsBomb Open Data 

Competitions: Premier League and Champions League

Season: 2015/2016

Objective:
Develop a football analytics platform using python, SQL, PostgreSQL, and Power BI

In [1]:
import pandas as pd

In [2]:
matches = pd.read_json(
    r"C:\Users\Sam\SportsAnalytics\ArsenalAnalytics\data\open-data-master\data\matches\2\27.json"
)

In [3]:
matches['home_team_name'] = matches['home_team'].apply(
    lambda x: x['home_team_name']
)

matches['away_team_name'] = matches['away_team'].apply(
    lambda x: x['away_team_name']
)

In [4]:
arsenal_matches = matches[
    (matches['home_team_name'] == 'Arsenal') |
     (matches['away_team_name'] == 'Arsenal')
    ]

In [5]:
arsenal_match_inventory = arsenal_matches[
    [
        'match_date',
        'home_team_name',
        'away_team_name',
        'home_score',
        'away_score',
        'match_id'
    ]
    ].copy()

In [6]:
arsenal_match_inventory['arsenal_goals'] = arsenal_match_inventory.apply(
    lambda row: row['home_score']
    if row['home_team_name'] == 'Arsenal'
    else row['away_score'],
    axis=1
)

In [7]:
arsenal_match_inventory['goals_conceded'] = arsenal_match_inventory.apply(
    lambda row: row['away_score']
    if row['home_team_name'] == 'Arsenal'
    else row['home_score'],
    axis=1
)

In [8]:
def get_result(row):

    if row['home_team_name'] == 'Arsenal':

        if row['home_score'] > row['away_score']:
            return 'Win'

        elif row['home_score'] < row['away_score']:
            return 'Loss'

        else:
            return 'Draw'

    else:

            if row['away_score'] > row['home_score']:
                return 'Win'

            elif row['away_score'] < row['home_score']:
                return 'Loss'

            else:
                return 'Draw'

In [9]:
arsenal_match_inventory['result'] = (
    arsenal_match_inventory.apply(
        get_result,
        axis=1
    )
)

In [10]:
arsenal_match_inventory['result'].value_counts()

result
Win     20
Draw    11
Loss     7
Name: count, dtype: int64

In [11]:
arsenal_match_inventory['points'] = (
    arsenal_match_inventory['result']
    .map({
        'Win': 3,
        'Draw': 1,
        'Loss': 0
    })
)

In [12]:
arsenal_match_inventory['points'].sum()

np.int64(71)

In [13]:
print("Matches:", len(arsenal_match_inventory))
print("Goals Scored:", arsenal_match_inventory['arsenal_goals'].sum())
print("Goals Conceded:", arsenal_match_inventory['goals_conceded'].sum())
print("Points:", arsenal_match_inventory['points'].sum())
      

Matches: 38
Goals Scored: 65
Goals Conceded: 36
Points: 71


In [14]:
arsenal_match_inventory.head()

,match_date,home_team_name,away_team_name,home_score,away_score,match_id,arsenal_goals,goals_conceded,result,points
0,2015-09-19,Chelsea,Arsenal,2,0,3754217,0,2,Loss,0
1,2015-12-13,Aston Villa,Arsenal,0,2,3754117,2,0,Win,3
2,2015-12-21,Arsenal,Manchester City,2,1,3754296,2,1,Win,3
3,2015-10-31,Swansea City,Arsenal,0,3,3753983,3,0,Win,3
4,2015-12-05,Arsenal,Sunderland,3,1,3754160,3,1,Win,3


## Load Event Data 

In [15]:
events = pd.read_json(
    r"C:\Users\Sam\SportsAnalytics\ArsenalAnalytics\data\open-data-master\data\events\3754217.json"
)

In [16]:
events.shape

(3732, 38)

In [17]:
events.head()

,id,index,period,timestamp,minute,second,type,possession,possession_team,play_pattern,...,clearance,foul_committed,out,interception,block,ball_recovery,miscontrol,injury_stoppage,bad_behaviour,substitution
0,9d86a178-3514-45d1-9d14-1372e846d17b,1,1,2026-08-19 00:00:00.000,0,0,"{'id': 35, 'name': 'Starting XI'}",1,"{'id': 33, 'name': 'Chelsea'}","{'id': 1, 'name': 'Regular Play'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,82acc213-90f3-4fee-8305-bd9e403cec42,2,1,2026-08-19 00:00:00.000,0,0,"{'id': 35, 'name': 'Starting XI'}",1,"{'id': 33, 'name': 'Chelsea'}","{'id': 1, 'name': 'Regular Play'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,d18f1a33-65ba-42e5-a9fd-bbfc709694e8,3,1,2026-08-19 00:00:00.000,0,0,"{'id': 18, 'name': 'Half Start'}",1,"{'id': 33, 'name': 'Chelsea'}","{'id': 1, 'name': 'Regular Play'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,49147091-455c-4372-8d56-96e1ca71d01a,4,1,2026-08-19 00:00:00.000,0,0,"{'id': 18, 'name': 'Half Start'}",1,"{'id': 33, 'name': 'Chelsea'}","{'id': 1, 'name': 'Regular Play'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5f9bdb6d-0379-4f42-9f53-29777f166db5,5,1,2026-08-19 00:00:00.622,0,0,"{'id': 30, 'name': 'Pass'}",2,"{'id': 1, 'name': 'Arsenal'}","{'id': 9, 'name': 'From Kick Off'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
events.columns.tolist()

['id',
 'index',
 'period',
 'timestamp',
 'minute',
 'second',
 'type',
 'possession',
 'possession_team',
 'play_pattern',
 'team',
 'duration',
 'tactics',
 'related_events',
 'player',
 'position',
 'location',
 'pass',
 'carry',
 'under_pressure',
 'ball_receipt',
 'counterpress',
 'dribble',
 'foul_won',
 'off_camera',
 'duel',
 'shot',
 'goalkeeper',
 'clearance',
 'foul_committed',
 'out',
 'interception',
 'block',
 'ball_recovery',
 'miscontrol',
 'injury_stoppage',
 'bad_behaviour',
 'substitution']

In [19]:
events['type'].head()

0    {'id': 35, 'name': 'Starting XI'}
1    {'id': 35, 'name': 'Starting XI'}
2     {'id': 18, 'name': 'Half Start'}
3     {'id': 18, 'name': 'Half Start'}
4           {'id': 30, 'name': 'Pass'}
Name: type, dtype: object

In [20]:
events.iloc[0]['type']

{'id': 35, 'name': 'Starting XI'}

In [21]:
events['events_type'] = events['type'].apply(
    lambda x: x['name']
)

In [22]:
events['events_type'].value_counts()

events_type
Pass                 1046
Ball Receipt*         963
Carry                 814
Pressure              320
Ball Recovery         106
Duel                   78
Dribble                52
Clearance              42
Block                  41
Goal Keeper            34
Dribbled Past          33
Shot                   33
Foul Committed         32
Foul Won               29
Miscontrol             27
Dispossessed           26
Interception           20
Substitution            6
Half Start              4
Injury Stoppage         4
Half End                4
Shield                  3
Bad Behaviour           3
Tactical Shift          3
Starting XI             2
Referee Ball-Drop       2
Player Off              1
Player On               1
Error                   1
Own Goal For            1
Own Goal Against        1
Name: count, dtype: int64

In [23]:
events['team_name'] = events['team'].apply(
    lambda x: x['name']
)

In [24]:
events['team_name'].value_counts()

team_name
Chelsea    2213
Arsenal    1519
Name: count, dtype: int64

In [25]:
arsenal_events = events[
    events['team_name'] == 'Arsenal'
    ]

In [26]:
arsenal_events.shape

(1519, 40)

In [27]:
arsenal_events['events_type'].value_counts()

events_type
Pass                 400
Ball Receipt*        357
Carry                295
Pressure             162
Ball Recovery         52
Duel                  45
Goal Keeper           24
Dribble               23
Clearance             21
Block                 20
Dribbled Past         19
Foul Committed        15
Foul Won              15
Dispossessed          14
Miscontrol            13
Interception          12
Shot                  10
Shield                 3
Tactical Shift         3
Substitution           3
Half Start             2
Injury Stoppage        2
Bad Behaviour          2
Half End               2
Starting XI            1
Player Off             1
Player On              1
Referee Ball-Drop      1
Own Goal Against       1
Name: count, dtype: int64

In [28]:
events.shape

(3732, 40)

In [29]:
events.iloc[0]['player']

nan

In [30]:
events[
    events['player'].notna()
    ].head()

,id,index,period,timestamp,minute,second,type,possession,possession_team,play_pattern,...,out,interception,block,ball_recovery,miscontrol,injury_stoppage,bad_behaviour,substitution,events_type,team_name
4,5f9bdb6d-0379-4f42-9f53-29777f166db5,5,1,2026-08-19 00:00:00.622,0,0,"{'id': 30, 'name': 'Pass'}",2,"{'id': 1, 'name': 'Arsenal'}","{'id': 9, 'name': 'From Kick Off'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass,Arsenal
5,a38327b4-a2c6-4282-bb8c-5988520fbd46,6,1,2026-08-19 00:00:01.008,0,1,"{'id': 42, 'name': 'Ball Receipt*'}",2,"{'id': 1, 'name': 'Arsenal'}","{'id': 9, 'name': 'From Kick Off'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ball Receipt*,Arsenal
6,e5d7db8c-d134-4392-a109-9f6efc7f7225,7,1,2026-08-19 00:00:01.008,0,1,"{'id': 30, 'name': 'Pass'}",2,"{'id': 1, 'name': 'Arsenal'}","{'id': 9, 'name': 'From Kick Off'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass,Arsenal
7,38484493-a9cc-4a15-8880-8123bc11de15,8,1,2026-08-19 00:00:02.508,0,2,"{'id': 42, 'name': 'Ball Receipt*'}",2,"{'id': 1, 'name': 'Arsenal'}","{'id': 9, 'name': 'From Kick Off'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ball Receipt*,Arsenal
8,7ec3ada2-d200-4f0b-ab86-f5a464dc4aa7,9,1,2026-08-19 00:00:02.508,0,2,"{'id': 43, 'name': 'Carry'}",2,"{'id': 1, 'name': 'Arsenal'}","{'id': 9, 'name': 'From Kick Off'}",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Carry,Arsenal


In [31]:
events[
    events['player'].notna()
    ].iloc[0]['player']

{'id': 3668, 'name': 'Theo Walcott'}

In [32]:
events['player_name'] = events['player'].apply(
    lambda x: x['name']
    if isinstance(x, dict)
    else None
)

In [33]:
events['player_name'].value_counts().head(20)

player_name
Francesc Fàbregas i Soler           276
Eden Hazard                         273
Nemanja Matić                       263
César Azpilicueta Tanco             245
Pedro Eliezer Rodríguez Ledesma     228
Aaron Ramsey                        206
Santiago Cazorla González           191
Diego da Silva Costa                172
Alexis Alejandro Sánchez Sánchez    166
Branislav Ivanović                  161
Oscar dos Santos Emboaba Júnior     155
Laurent Koscielny                   144
Héctor Bellerín Moruno              132
Mesut Özil                          126
Ignacio Monreal Eraso               119
Kurt Happy Zouma                    118
Gary Cahill                         118
Theo Walcott                        101
Ramires Santos do Nascimento         88
Petr Čech                            83
Name: count, dtype: int64

In [34]:
events.columns.tolist()

['id',
 'index',
 'period',
 'timestamp',
 'minute',
 'second',
 'type',
 'possession',
 'possession_team',
 'play_pattern',
 'team',
 'duration',
 'tactics',
 'related_events',
 'player',
 'position',
 'location',
 'pass',
 'carry',
 'under_pressure',
 'ball_receipt',
 'counterpress',
 'dribble',
 'foul_won',
 'off_camera',
 'duel',
 'shot',
 'goalkeeper',
 'clearance',
 'foul_committed',
 'out',
 'interception',
 'block',
 'ball_recovery',
 'miscontrol',
 'injury_stoppage',
 'bad_behaviour',
 'substitution',
 'events_type',
 'team_name',
 'player_name']

In [35]:
arsenal_events = events[
    events['team_name'] == 'Arsenal'
    ]

In [36]:
arsenal_events['player_name'].value_counts().head(15)

player_name
Aaron Ramsey                        206
Santiago Cazorla González           191
Alexis Alejandro Sánchez Sánchez    166
Laurent Koscielny                   144
Héctor Bellerín Moruno              132
Mesut Özil                          126
Ignacio Monreal Eraso               119
Theo Walcott                        101
Petr Čech                            83
Gabriel Armando de Abreu             65
Francis Joseph Coquelin              62
Calum Chambers                       57
Alex Oxlade-Chamberlain              33
Olivier Giroud                       25
Name: count, dtype: int64

In [37]:
arsenal_passes = arsenal_events[
    arsenal_events['events_type'] == 'Pass'
    ]

In [38]:
arsenal_passes['player_name'].value_counts().head(10)

player_name
Santiago Cazorla González           62
Aaron Ramsey                        47
Ignacio Monreal Eraso               43
Héctor Bellerín Moruno              43
Laurent Koscielny                   35
Mesut Özil                          33
Petr Čech                           32
Alexis Alejandro Sánchez Sánchez    30
Gabriel Armando de Abreu            18
Theo Walcott                        17
Name: count, dtype: int64

In [39]:
arsenal_shots = arsenal_events[
    arsenal_events['events_type'] == 'Shot'
    ]

In [40]:
arsenal_shots['player_name'].value_counts().head(10)

player_name
Theo Walcott                        4
Aaron Ramsey                        3
Alexis Alejandro Sánchez Sánchez    3
Name: count, dtype: int64

In [41]:
arsenal_shots.iloc[0]['shot']

{'statsbomb_xg': 0.0388317,
 'end_location': [113.7, 41.9],
 'first_time': True,
 'technique': {'id': 93, 'name': 'Normal'},
 'body_part': {'id': 38, 'name': 'Left Foot'},
 'type': {'id': 87, 'name': 'Open Play'},
 'outcome': {'id': 96, 'name': 'Blocked'},
 'freeze_frame': [{'location': [107.2, 45.2],
   'player': {'id': 3957, 'name': 'César Azpilicueta Tanco'},
   'position': {'id': 6, 'name': 'Left Back'},
   'teammate': False},
  {'location': [97.5, 43.8],
   'player': {'id': 3381, 'name': 'Nemanja Matić'},
   'position': {'id': 11, 'name': 'Left Defensive Midfield'},
   'teammate': False},
  {'location': [100.9, 36.9],
   'player': {'id': 3958, 'name': 'Pedro Eliezer Rodríguez Ledesma'},
   'position': {'id': 17, 'name': 'Right Wing'},
   'teammate': False},
  {'location': [108.8, 30.3],
   'player': {'id': 3645, 'name': 'Gary Cahill'},
   'position': {'id': 5, 'name': 'Left Center Back'},
   'teammate': False},
  {'location': [113.3, 41.9],
   'player': {'id': 3456, 'name': 'Kurt 

In [42]:
arsenal_shots.iloc[0]['shot']

{'statsbomb_xg': 0.0388317,
 'end_location': [113.7, 41.9],
 'first_time': True,
 'technique': {'id': 93, 'name': 'Normal'},
 'body_part': {'id': 38, 'name': 'Left Foot'},
 'type': {'id': 87, 'name': 'Open Play'},
 'outcome': {'id': 96, 'name': 'Blocked'},
 'freeze_frame': [{'location': [107.2, 45.2],
   'player': {'id': 3957, 'name': 'César Azpilicueta Tanco'},
   'position': {'id': 6, 'name': 'Left Back'},
   'teammate': False},
  {'location': [97.5, 43.8],
   'player': {'id': 3381, 'name': 'Nemanja Matić'},
   'position': {'id': 11, 'name': 'Left Defensive Midfield'},
   'teammate': False},
  {'location': [100.9, 36.9],
   'player': {'id': 3958, 'name': 'Pedro Eliezer Rodríguez Ledesma'},
   'position': {'id': 17, 'name': 'Right Wing'},
   'teammate': False},
  {'location': [108.8, 30.3],
   'player': {'id': 3645, 'name': 'Gary Cahill'},
   'position': {'id': 5, 'name': 'Left Center Back'},
   'teammate': False},
  {'location': [113.3, 41.9],
   'player': {'id': 3456, 'name': 'Kurt 

In [43]:
arsenal_shots['team_name'].value_counts()

team_name
Arsenal    10
Name: count, dtype: int64

In [44]:
arsenal_shots[['player_name', 'team_name']].head(10)

,player_name,team_name
95,Aaron Ramsey,Arsenal
442,Alexis Alejandro Sánchez Sánchez,Arsenal
1096,Theo Walcott,Arsenal
1355,Aaron Ramsey,Arsenal
1831,Alexis Alejandro Sánchez Sánchez,Arsenal
2237,Alexis Alejandro Sánchez Sánchez,Arsenal
2469,Theo Walcott,Arsenal
3007,Theo Walcott,Arsenal
3578,Theo Walcott,Arsenal
3582,Aaron Ramsey,Arsenal


In [45]:
arsenal_shots['player_name'].value_counts()

player_name
Theo Walcott                        4
Aaron Ramsey                        3
Alexis Alejandro Sánchez Sánchez    3
Name: count, dtype: int64

In [46]:
len(arsenal_shots)

10

In [47]:
len(
    events[
        events['events_type'] == 'Shot'
        ]
)

33

In [48]:
arsenal_shots['xg'] = arsenal_shots['shot'].apply(
    lambda x: x.get('statsbomb_xg')
)

In [49]:
arsenal_shots['xg'].sum()

np.float64(0.8375268513)

In [50]:
arsenal_shots[
    ['player_name', 'xg']
    ].sort_values(
        'xg',
        ascending=False
    ).head(10)

,player_name,xg
3578,Theo Walcott,0.338819
3582,Aaron Ramsey,0.167530
2237,Alexis Alejandro Sánchez Sánchez,0.097416
1831,Alexis Alejandro Sánchez Sánchez,0.078176
1096,Theo Walcott,0.049741
95,Aaron Ramsey,0.038832
3007,Theo Walcott,0.025205
1355,Aaron Ramsey,0.023207
2469,Theo Walcott,0.011942
442,Alexis Alejandro Sánchez Sánchez,0.006660


In [51]:
arsenal_shots['shot_outcome'] = arsenal_shots['shot'].apply(
    lambda x: x['outcome']['name']
)

In [52]:
arsenal_shots['shot_outcome'].value_counts()

shot_outcome
Off T      4
Blocked    3
Saved      2
Wayward    1
Name: count, dtype: int64

In [53]:
arsenal_shots[
    arsenal_shots['shot_outcome'] == 'Goal'
    ][
    ['player_name', 'minute']
    ]

,player_name,minute


In [54]:
arsenal_passes['player_name'].value_counts()

player_name
Santiago Cazorla González           62
Aaron Ramsey                        47
Ignacio Monreal Eraso               43
Héctor Bellerín Moruno              43
Laurent Koscielny                   35
Mesut Özil                          33
Petr Čech                           32
Alexis Alejandro Sánchez Sánchez    30
Gabriel Armando de Abreu            18
Theo Walcott                        17
Francis Joseph Coquelin             14
Calum Chambers                      14
Alex Oxlade-Chamberlain              8
Olivier Giroud                       4
Name: count, dtype: int64

In [55]:
print("Matches:", len(arsenal_match_inventory))
print("Goals Scored:", arsenal_match_inventory['arsenal_goals'].sum())
print("Goals Conceded:", arsenal_match_inventory['goals_conceded'].sum())
print("Points:", arsenal_match_inventory['points'].sum())

Matches: 38
Goals Scored: 65
Goals Conceded: 36
Points: 71


# Arsenal Match Report

## Team Statistics

In [56]:
len(arsenal_events)

1519

In [57]:
arsenal_events['events_type'].value_counts()

events_type
Pass                 400
Ball Receipt*        357
Carry                295
Pressure             162
Ball Recovery         52
Duel                  45
Goal Keeper           24
Dribble               23
Clearance             21
Block                 20
Dribbled Past         19
Foul Committed        15
Foul Won              15
Dispossessed          14
Miscontrol            13
Interception          12
Shot                  10
Shield                 3
Tactical Shift         3
Substitution           3
Half Start             2
Injury Stoppage        2
Bad Behaviour          2
Half End               2
Starting XI            1
Player Off             1
Player On              1
Referee Ball-Drop      1
Own Goal Against       1
Name: count, dtype: int64

In [58]:
arsenal_passes = arsenal_events[
    arsenal_events['events_type'] == 'Pass'
    ]
len(arsenal_passes)

400

In [59]:
len(arsenal_shots)

10

In [60]:
arsenal_shots['xg'].sum()

np.float64(0.8375268513)

## Top Passers

In [61]:
arsenal_passes['player_name'].value_counts().head(10)

player_name
Santiago Cazorla González           62
Aaron Ramsey                        47
Ignacio Monreal Eraso               43
Héctor Bellerín Moruno              43
Laurent Koscielny                   35
Mesut Özil                          33
Petr Čech                           32
Alexis Alejandro Sánchez Sánchez    30
Gabriel Armando de Abreu            18
Theo Walcott                        17
Name: count, dtype: int64

## Top Shooters

In [62]:
arsenal_shots['player_name'].value_counts().head(10)

player_name
Theo Walcott                        4
Aaron Ramsey                        3
Alexis Alejandro Sánchez Sánchez    3
Name: count, dtype: int64

## Highest Quality Chances

In [63]:
arsenal_shots[
    ['player_name','minute','xg']
    ].sort_values(
        by='xg',
        ascending=False
    ).head(10)
    

,player_name,minute,xg
3578,Theo Walcott,91,0.338819
3582,Aaron Ramsey,92,0.167530
2237,Alexis Alejandro Sánchez Sánchez,59,0.097416
1831,Alexis Alejandro Sánchez Sánchez,49,0.078176
1096,Theo Walcott,28,0.049741
95,Aaron Ramsey,1,0.038832
3007,Theo Walcott,77,0.025205
1355,Aaron Ramsey,34,0.023207
2469,Theo Walcott,63,0.011942
442,Alexis Alejandro Sánchez Sánchez,11,0.006660


## Shot Outcomes

In [64]:
arsenal_shots['shot_outcome'] = arsenal_shots['shot'].apply(
    lambda x: x['outcome']['name']
)

In [65]:
arsenal_shots['shot_outcome'].value_counts()

shot_outcome
Off T      4
Blocked    3
Saved      2
Wayward    1
Name: count, dtype: int64

In [66]:
arsenal_shots[
    arsenal_shots['shot_outcome'] == 'Goal'
    ][
    ['player_name','minute']
    ]

,player_name,minute


## Most Involved Player

In [67]:
arsenal_events['player_name'].value_counts().head(10)

player_name
Aaron Ramsey                        206
Santiago Cazorla González           191
Alexis Alejandro Sánchez Sánchez    166
Laurent Koscielny                   144
Héctor Bellerín Moruno              132
Mesut Özil                          126
Ignacio Monreal Eraso               119
Theo Walcott                        101
Petr Čech                            83
Gabriel Armando de Abreu             65
Name: count, dtype: int64

## Match Findings

Chelsea 1 Arsenal 0 - 2015/2016 Season Game Findings

Arsenal had 400 passes, 10 shots, and an expected goal percentage of 0.84

Santi Cazorla led the way with 62 passes followed by Aaron Ramsey with 47

There was only 3 players with shots recorded and it is Theo Walcott leading the way with 4 shots followed by Ramsey and Alexis Sanchez with 3 shots each

Walcott and Ramsey were the playest who created the highest quality chances

Aaron Ramsey and Santi Cazorla were the most involved in the game being the only players to record over 190 events with the ball

## Season - Wide Event Analytics

## Objective

Load all Arsenal event data from the 2015/2016 Premier League season and create a season-level event dataset.

In [68]:
arsenal_match_inventory.head()

,match_date,home_team_name,away_team_name,home_score,away_score,match_id,arsenal_goals,goals_conceded,result,points
0,2015-09-19,Chelsea,Arsenal,2,0,3754217,0,2,Loss,0
1,2015-12-13,Aston Villa,Arsenal,0,2,3754117,2,0,Win,3
2,2015-12-21,Arsenal,Manchester City,2,1,3754296,2,1,Win,3
3,2015-10-31,Swansea City,Arsenal,0,3,3753983,3,0,Win,3
4,2015-12-05,Arsenal,Sunderland,3,1,3754160,3,1,Win,3


In [69]:
arsenal_match_inventory['match_id'].nunique()

38

In [70]:
match_ids = arsenal_match_inventory['match_id'].tolist()

In [71]:
match_ids[:5]

[3754217, 3754117, 3754296, 3753983, 3754160]

In [72]:
season_events = []

In [73]:
for match_id in match_ids:

    file_path = (
        rf"C:\Users\Sam\SportsAnalytics\ArsenalAnalytics\data\open-data-master\data\events\{match_id}.json"
    )

    match_events = pd.read_json(file_path)

    match_events['match_id'] = match_id

    season_events.append(match_events)

In [74]:
arsenal_season_events = pd.concat(
    season_events,
    ignore_index=True
)

In [75]:
arsenal_season_events.shape

(140328, 40)

In [76]:
arsenal_season_events['match_id'].nunique()

38

In [77]:
arsenal_season_events['team_name'] = (
    arsenal_season_events['team']
    .apply(
        lambda x: x['name']
        if isinstance(x, dict)
        else None
    )
)

In [78]:
arsenal_season_events = arsenal_season_events[
    arsenal_season_events['team_name'] == 'Arsenal'
    ]

In [79]:
arsenal_season_events.shape

(78070, 41)

In [80]:
arsenal_season_events['player_name'] = (
    arsenal_season_events['player']
    .apply(
        lambda x: x['name']
        if isinstance(x, dict)
        else None
    )
)

In [81]:
arsenal_season_events['player_name'].value_counts().head(20)

player_name
Mesut Özil                          8413
Aaron Ramsey                        8051
Ignacio Monreal Eraso               6539
Alexis Alejandro Sánchez Sánchez    6455
Héctor Bellerín Moruno              6412
Laurent Koscielny                   5170
Olivier Giroud                      4139
Santiago Cazorla González           4106
Francis Joseph Coquelin             3862
Per Mertesacker                     3443
Gabriel Armando de Abreu            3272
Petr Čech                           2562
Mohamed Naser Elsayed Elneny        2401
Alex Oxlade-Chamberlain             2094
Mathieu Flamini                     1986
Theo Walcott                        1841
Joel Nathaniel Campbell Samuels     1733
Alex Iwobi                          1485
Daniel Nii Tackie Mensah Welbeck     976
Kieran Gibbs                         841
Name: count, dtype: int64

In [82]:
top_players = (
    arsenal_season_events['player_name']
    .value_counts()
    .head(20)
)

top_players

player_name
Mesut Özil                          8413
Aaron Ramsey                        8051
Ignacio Monreal Eraso               6539
Alexis Alejandro Sánchez Sánchez    6455
Héctor Bellerín Moruno              6412
Laurent Koscielny                   5170
Olivier Giroud                      4139
Santiago Cazorla González           4106
Francis Joseph Coquelin             3862
Per Mertesacker                     3443
Gabriel Armando de Abreu            3272
Petr Čech                           2562
Mohamed Naser Elsayed Elneny        2401
Alex Oxlade-Chamberlain             2094
Mathieu Flamini                     1986
Theo Walcott                        1841
Joel Nathaniel Campbell Samuels     1733
Alex Iwobi                          1485
Daniel Nii Tackie Mensah Welbeck     976
Kieran Gibbs                         841
Name: count, dtype: int64

## Season Passing Analysis 

In [83]:
arsenal_season_events['event_type'] = (
    arsenal_season_events['type']
    .apply(
        lambda x: x['name']
        if isinstance(x, dict)
        else None
    )
)

In [84]:
arsenal_season_passes = arsenal_season_events[
    arsenal_season_events['event_type'] == 'Pass'
    ]

In [85]:
len(arsenal_season_passes)

22709

In [86]:
arsenal_season_passes['player_name'].value_counts().head(20)

player_name
Mesut Özil                          2511
Ignacio Monreal Eraso               2259
Aaron Ramsey                        2197
Héctor Bellerín Moruno              2094
Laurent Koscielny                   1671
Alexis Alejandro Sánchez Sánchez    1488
Santiago Cazorla González           1300
Per Mertesacker                     1238
Francis Joseph Coquelin             1104
Gabriel Armando de Abreu            1074
Petr Čech                           1036
Olivier Giroud                       825
Mohamed Naser Elsayed Elneny         744
Mathieu Flamini                      627
Alex Oxlade-Chamberlain              495
Alex Iwobi                           376
Joel Nathaniel Campbell Samuels      350
Theo Walcott                         333
Kieran Gibbs                         219
Calum Chambers                       190
Name: count, dtype: int64

In [87]:
arsenal_season_shots=arsenal_season_events[
    arsenal_season_events['event_type'] == 'Shot'
    ]

In [88]:
len(arsenal_season_shots)

583

In [89]:
arsenal_season_shots['player_name'].value_counts().head(20)

player_name
Alexis Alejandro Sánchez Sánchez    111
Olivier Giroud                      103
Aaron Ramsey                         67
Mesut Özil                           49
Theo Walcott                         46
Santiago Cazorla González            24
Alex Oxlade-Chamberlain              23
Laurent Koscielny                    20
Joel Nathaniel Campbell Samuels      20
Daniel Nii Tackie Mensah Welbeck     20
Alex Iwobi                           15
Héctor Bellerín Moruno               13
Ignacio Monreal Eraso                13
Mohamed Naser Elsayed Elneny         13
Gabriel Armando de Abreu             12
Francis Joseph Coquelin               9
Per Mertesacker                       8
Mathieu Flamini                       7
Kieran Gibbs                          5
Calum Chambers                        2
Name: count, dtype: int64

In [90]:
arsenal_season_shots['xg'] = (
    arsenal_season_shots['shot']
    .apply(
        lambda x: x.get('statsbomb_xg')
        if isinstance(x, dict)
        else None
    )
)

In [91]:
arsenal_season_shots.groupby(
    'player_name'
)['xg'].sum().sort_values(
    ascending=False
).head(20)

player_name
Olivier Giroud                      13.017703
Alexis Alejandro Sánchez Sánchez    11.263074
Aaron Ramsey                         7.372339
Theo Walcott                         6.700536
Mesut Özil                           5.966241
Laurent Koscielny                    3.202166
Daniel Nii Tackie Mensah Welbeck     3.023732
Santiago Cazorla González            1.784201
Joel Nathaniel Campbell Samuels      1.756452
Alex Iwobi                           1.723545
Alex Oxlade-Chamberlain              1.475736
Per Mertesacker                      1.236623
Ignacio Monreal Eraso                1.202076
Gabriel Armando de Abreu             1.041712
Mohamed Naser Elsayed Elneny         0.975098
Mathieu Flamini                      0.919226
Kieran Gibbs                         0.674482
Héctor Bellerín Moruno               0.614905
Mikel Arteta Amatriain               0.372364
Jack Wilshere                        0.314301
Name: xg, dtype: float64

## Season Goal Scoring Analysis

In [92]:
arsenal_season_shots['shot_outcome'] = (
    arsenal_season_shots['shot']
    .apply(
        lambda x: x['outcome']['name']
    )
)

In [93]:
arsenal_goals = arsenal_season_shots[
    arsenal_season_shots['shot_outcome'] == 'Goal'
    ]

In [94]:
arsenal_goals['player_name'].value_counts()

player_name
Olivier Giroud                      16
Alexis Alejandro Sánchez Sánchez    13
Mesut Özil                           6
Aaron Ramsey                         5
Theo Walcott                         5
Laurent Koscielny                    4
Daniel Nii Tackie Mensah Welbeck     4
Joel Nathaniel Campbell Samuels      3
Alex Iwobi                           2
Kieran Gibbs                         1
Gabriel Armando de Abreu             1
Héctor Bellerín Moruno               1
Alex Oxlade-Chamberlain              1
Name: count, dtype: int64

## Arsenal 2015/16 Attack Report

Mesut Ozil was the most involved player throughout the season with 8413 events with the ball

Mesut Ozil lead the team with most passes in the season with 2511

Alexis Sanchez had the most shots with 111 

Olivier Giroud had the highest expected goal percentage [xg] at 13.02

Giroud also lead the Gunners with 16 goals across the season followed by Alexis Sanchez with 13. These are the only 2 Arsenal players with double digit
goals on the season.

In [95]:
arsenal_goals['player_name'].value_counts()

player_name
Olivier Giroud                      16
Alexis Alejandro Sánchez Sánchez    13
Mesut Özil                           6
Aaron Ramsey                         5
Theo Walcott                         5
Laurent Koscielny                    4
Daniel Nii Tackie Mensah Welbeck     4
Joel Nathaniel Campbell Samuels      3
Alex Iwobi                           2
Kieran Gibbs                         1
Gabriel Armando de Abreu             1
Héctor Bellerín Moruno               1
Alex Oxlade-Chamberlain              1
Name: count, dtype: int64

In [96]:
goals_by_player = (
    arsenal_goals['player_name']
    .value_counts()
    .reset_index()
)

In [97]:
goals_by_player.columns = [
    'player_name',
    'goals'
]

In [98]:
goals_by_player.head()

,player_name,goals
0,Olivier Giroud,16
1,Alexis Alejandro Sánchez Sánchez,13
2,Mesut Özil,6
3,Aaron Ramsey,5
4,Theo Walcott,5


In [99]:
arsenal_season_shots[['player_name', 'xg']].head()

,player_name,xg
95,Aaron Ramsey,0.038832
442,Alexis Alejandro Sánchez Sánchez,0.006660
1096,Theo Walcott,0.049741
1355,Aaron Ramsey,0.023207
1831,Alexis Alejandro Sánchez Sánchez,0.078176


In [100]:
xg_by_player

NameError: name 'xg_by_player' is not defined

In [ ]:
type(xg_by_player)

In [ ]:
xg_by_player = (
    arsenal_season_shots
    .groupby('player_name')['xg']
    .sum()
    .reset_index()
)

In [ ]:
xg_by_player.columns = [
    'player_name',
    'xg'
]

In [ ]:
type(xg_by_player)

In [ ]:
xg_by_player.head()

In [ ]:
goals_vs_xg = goals_by_player.merge(
    xg_by_player,
    on='player_name',
    how='left'
)

In [ ]:
goals_vs_xg['goals_minus_xg'] = (
    goals_vs_xg['goals']
    - goals_vs_xg['xg']
)

In [ ]:
goals_vs_xg.head()

In [ ]:
goals_vs_xg.sort_values(
    by='goals_minus_xg',
    ascending=False
).head(10)

In [ ]:
goals_vs_xg.sort_values(
    by='goals_minus_xg',
    ascending=True
).head(10)

## Goals vs Expected Goals Analysis

Olivier Giroud was the most clinical finisher for Arsenal this season scoring almost 3 goals over expected.

Aaron Ramsey was expected to finish the season with 7 goals but finished with 5. Leading to some missed opportunities for the Gunners

## Shots Summary

In [ ]:
shots_by_player = (
    arsenal_season_shots['player_name']
    .value_counts()
    .reset_index()
)

In [ ]:
shots_by_player.columns = [
    'player_name',
    'shots'
]

In [ ]:
shots_by_player.head()

In [ ]:
goals_by_player.head()

In [ ]:
shot_efficiency = shots_by_player.merge(
    goals_by_player,
    on='player_name',
    how='left'
)

In [ ]:
shot_efficiency['goals'] = (
    shot_efficiency['goals']
    .fillna(0)
)

In [ ]:
shot_efficiency['shots_per_goal'] = (
    shot_efficiency['shots']
    / shot_efficiency['goals']
)

In [ ]:
shot_efficiency_goals = shot_efficiency[
    shot_efficiency['goals'] > 0
    ]

## Most Efficient Goal Scorer

In [ ]:
shot_efficiency_goals.sort_values(
    by='shots_per_goal',
    ascending=True
)

## Least Efficient Goal Scorer

In [ ]:
shot_efficiency_goals.sort_values(
    by='shots_per_goal',
    ascending=False
)

## Shot Efficiency Analysis

Laurent Koscielny was the most efficient Arsenal goal scorer, scoring a goal once every 5 shots but only had 20 on the season

Alex Oxlade-Chamberlain was the least efficient scoring only 1 goal on 23 shots

In [ ]:
player_summary = (
    arsenal_season_events['player_name']
    .value_counts()
    .reset_index()
)

In [ ]:
player_summary.columns = [
    'player_name',
    'total_events'
]

In [ ]:
player_summary.head()

In [ ]:
passes_by_player = (
    arsenal_season_passes['player_name']
    .value_counts()
    .reset_index()
)

In [ ]:
passes_by_player.columns = [
    'player_name',
    'passes'
]

In [ ]:
passes_by_player.head()

In [ ]:
player_summary.head()

In [ ]:
player_summary = player_summary.merge(
    passes_by_player,
    on='player_name',
    how='left'
)

In [ ]:
player_summary.head()

In [ ]:
shots_by_player = (
    arsenal_season_shots['player_name']
    .value_counts()
    .reset_index()
)
    

In [ ]:
shots_by_player.columns = [
    'player_name',
    'shots'
]

In [ ]:
player_summary = player_summary.merge(
    shots_by_player,
    on='player_name',
    how='left'
)

In [ ]:
player_summary.head()

In [ ]:
player_summary = player_summary.merge(
    goals_by_player,
    on='player_name',
    how='left'
)

In [ ]:
player_summary.head()

In [ ]:
player_summary = player_summary.merge(
    xg_by_player,
    on='player_name',
    how='left'
)

In [ ]:
player_summary.head()

In [ ]:
player_summary = player_summary.fillna(0)

In [ ]:
player_summary['goals_minus_xg'] = (
    player_summary['goals']
    - player_summary['xg']
)

In [ ]:
player_summary['shots_per_goal'] = (
    player_summary['shots']
    / player_summary['goals']
)

In [ ]:
import numpy as np

player_summary['shots_per_goal'] = (
    player_summary['shots_per_goal']
    .replace([np.inf, -np.inf], np.nan)
)

In [ ]:
player_summary.sort_values(
    by='goals',
    ascending=False
).head(15)

In [ ]:
player_summary.columns.tolist()

In [ ]:
player_summary = player_summary.drop(
    columns = ['shots_per_goals']
)

In [ ]:
player_summary.head(20)

In [ ]:
player_summary = player_summary.fillna(0)

In [ ]:
player_summary.head(20)

In [ ]:
player_summary['xg'] = (
    player_summary['xg']
    .round(2)
)

player_summary['goals_minus_xg'] = (
    player_summary['goals_minus_xg']
    .round(2)
)

player_summary['shots_per_goal'] = (
    player_summary['shots_per_goal']
    .round(2)
)

In [ ]:
player_summary['goals_minus_xg'] = (
    player_summary['goals']
    - player_summary['xg']
)

In [ ]:
player_summary['shots_per_goal'] = (
    player_summary['shots']
    / player_summary['goals']
)

In [ ]:
import numpy as np

player_summary['shots_per_goal'] = (
    player_summary['shots_per_goal']
    .replace([np.inf, -np.inf], np.nan)
)

In [ ]:
player_summary.sort_values(
    by='goals',
    ascending=False
).head(15)

In [ ]:
player_summary.shape

In [ ]:
player_summary.to_csv(

    r"C:\Users\Sam\SportsAnalytics\ArsenalAnalytics\exports\arsenal_player_summary.csv",
        index=False
)

# Team Summary

In [101]:
team_summary = pd.DataFrame({
    'matches_played': [38],
    'wins': [20],
    'draws': [11],
    'losses': [7],
    'goals_scored': [65],
    'goals_conceded': [36],
    'goal_difference': [29],
    'points': [71]
})

In [102]:
team_summary

,matches_played,wins,draws,losses,goals_scored,goals_conceded,goal_difference,points
0,38,20,11,7,65,36,29,71


In [121]:
team_summary.to_csv(

    r"C:\Users\Sam\SportsAnalytics\ArsenalAnalytics\exports\arsenal_team_summary.csv",
    index=False
)

In [104]:
arsenal_match_inventory.head()

,match_date,home_team_name,away_team_name,home_score,away_score,match_id,arsenal_goals,goals_conceded,result,points
0,2015-09-19,Chelsea,Arsenal,2,0,3754217,0,2,Loss,0
1,2015-12-13,Aston Villa,Arsenal,0,2,3754117,2,0,Win,3
2,2015-12-21,Arsenal,Manchester City,2,1,3754296,2,1,Win,3
3,2015-10-31,Swansea City,Arsenal,0,3,3753983,3,0,Win,3
4,2015-12-05,Arsenal,Sunderland,3,1,3754160,3,1,Win,3


In [105]:
arsenal_match_inventory.columns.tolist()

['match_date',
 'home_team_name',
 'away_team_name',
 'home_score',
 'away_score',
 'match_id',
 'arsenal_goals',
 'goals_conceded',
 'result',
 'points']

In [106]:
arsenal_match_inventory.head()

,match_date,home_team_name,away_team_name,home_score,away_score,match_id,arsenal_goals,goals_conceded,result,points
0,2015-09-19,Chelsea,Arsenal,2,0,3754217,0,2,Loss,0
1,2015-12-13,Aston Villa,Arsenal,0,2,3754117,2,0,Win,3
2,2015-12-21,Arsenal,Manchester City,2,1,3754296,2,1,Win,3
3,2015-10-31,Swansea City,Arsenal,0,3,3753983,3,0,Win,3
4,2015-12-05,Arsenal,Sunderland,3,1,3754160,3,1,Win,3


In [107]:
arsenal_match_inventory['match_date'] = pd.to_datetime(
    arsenal_match_inventory['match_date']
)

arsenal_match_inventory['month'] = (
    arsenal_match_inventory['match_date']
    .dt.strftime('%B')
)

In [108]:
arsenal_match_inventory[
    ['match_date', 'month']
    ].head()

,match_date,month
0,2015-09-19,September
1,2015-12-13,December
2,2015-12-21,December
3,2015-10-31,October
4,2015-12-05,December


In [109]:
monthly_summary = (
    arsenal_match_inventory
    .groupby('month')
    .agg({
        'points': 'sum',
        'arsenal_goals': 'sum',
        'goals_conceded': 'sum'
    })
    .reset_index()
)
        

In [110]:
monthly_summary

,month,points,arsenal_goals,goals_conceded
0,April,12,11,4
1,August,7,3,3
2,December,12,9,6
3,February,7,6,4
4,January,5,4,4
5,March,4,5,4
6,May,4,6,2
7,November,2,3,4
8,October,12,11,1
9,September,6,7,4


In [111]:
month_sort = {
    'August': 1,
    'September': 2,
    'October': 3,
    'November': 4,
    'December': 5,
    'January': 6,
    'February': 7,
    'March': 8,
    'April': 9,
    'May': 10
}

In [113]:
monthly_summary['month_sort'] = (
    monthly_summary['month']
    .map(month_sort)
)

In [115]:
monthly_summary = monthly_summary.sort_values(
    by='month_sort'
)

In [116]:
monthly_summary

,month,points,arsenal_goals,goals_conceded,month_sort
1,August,7,3,3,1
9,September,6,7,4,2
8,October,12,11,1,3
7,November,2,3,4,4
2,December,12,9,6,5
4,January,5,4,4,6
3,February,7,6,4,7
5,March,4,5,4,8
0,April,12,11,4,9
6,May,4,6,2,10


In [117]:
monthly_summary = monthly_summary.reset_index(
    drop=True
)

In [118]:
monthly_summary

,month,points,arsenal_goals,goals_conceded,month_sort
0,August,7,3,3,1
1,September,6,7,4,2
2,October,12,11,1,3
3,November,2,3,4,4
4,December,12,9,6,5
5,January,5,4,4,6
6,February,7,6,4,7
7,March,4,5,4,8
8,April,12,11,4,9
9,May,4,6,2,10


In [120]:
monthly_summary.to_csv(

    r"C:\Users\Sam\SportsAnalytics\ArsenalAnalytics\exports\monthly_summary.csv",
    index=False
)

In [122]:
arsenal_match_inventory.columns.tolist()

['match_date',
 'home_team_name',
 'away_team_name',
 'home_score',
 'away_score',
 'match_id',
 'arsenal_goals',
 'goals_conceded',
 'result',
 'points',
 'month']

In [123]:
arsenal_match_inventory['location'] = (
    arsenal_match_inventory['home_team_name']
    .apply(
        lambda x: 'Home'
        if x == 'Arsenal'
        else 'Away'
    )
)

In [124]:
arsenal_match_inventory[
    ['home_team_name',
     'away_team_name',
     'location']
    ].head()
     

,home_team_name,away_team_name,location
0,Chelsea,Arsenal,Away
1,Aston Villa,Arsenal,Away
2,Arsenal,Manchester City,Home
3,Swansea City,Arsenal,Away
4,Arsenal,Sunderland,Home


In [133]:
home_away_summary = (
    (
        arsenal_match_inventory
    )
    .groupby('location')
    .agg({
        'points': 'sum',
        'arsenal_goals': 'sum',
        'goals_conceded': 'sum',
        'match_id': 'count'
    })
    .reset_index()
)


In [134]:
home_away_summary.columns = [
    'location',
    'points',
    'goals_scored',
    'goals_conceded',
    'matches'
]

In [135]:
home_away_summary

,location,points,goals_scored,goals_conceded,matches
0,Away,31,34,25,19
1,Home,40,31,11,19


In [136]:
home_away_summary.to_csv(

    r"C:\Users\Sam\SportsAnalytics\ArsenalAnalytics\exports\home_away_summary.csv",
    index=False
)

In [138]:
arsenal_match_inventory.to_csv(

    r"C:\Users\Sam\SportsAnalytics\ArsenalAnalytics\exports\arsenal_match_inventory.csv",
    index=False
)